# 商品明细导入 MySQL

将商品交易明细和待复核描述明细分别导入 product 和 product_pending 表。

In [1]:
import pandas as pd
from sqlalchemy import URL, create_engine, text

from mysql_import import write_dataframe_to_mysql
import os
from pathlib import Path
from dotenv import load_dotenv

In [2]:
load_dotenv(Path.cwd() / "../config/.env", interpolate=False, encoding="utf-8-sig")
mysql_user = os.getenv("UCI_MYSQL_USER")
mysql_password = os.getenv("UCI_MYSQL_PASSWORD")

if not mysql_user or not mysql_password:
    raise RuntimeError("Set UCI_MYSQL_USER and UCI_MYSQL_PASSWORD in the kernel environment or in ../config/.env.")

# 创建数据库连接地址
url = URL.create(
    drivername="mysql+pymysql",
    username=mysql_user,
    password=mysql_password,
    host="127.0.0.1",
    port=3306,
    database="data_10month",
    query={"charset": "utf8mb4"}
)

# 创建连接引擎
engine = create_engine(url, pool_pre_ping=True)

# 检查连接的数据库和 MySQL 版本
with engine.connect() as conn:
    result = conn.execute(
        text("SELECT DATABASE(), VERSION()")
    )
    print(result.fetchone())

('data_10month', '8.4.11')


In [3]:
df1 = pd.read_csv('../data/processed/data_10_product.csv')
df2 = pd.read_csv('../data/processed/data_10_product_pending.csv')

In [4]:
write_dataframe_to_mysql(df1, "product", engine)
write_dataframe_to_mysql(df2, "product_pending", engine)

586